In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
OUT_DIR = "./outputs_localsoil"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import existing implementations
from cgt_xai.models import get_model
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.shapley_shubik import exact_shapley_shubik
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    # 1. Load exactly the four arrays (fail if keys missing)
    try:
        X_train = d["X_train"].astype(float)
        X_test  = d["X_test"].astype(float)
        y_train = d["y_train"].astype(int).ravel() # Ensure 1D
        y_test  = d["y_test"].astype(int).ravel()
    except KeyError as e:
        raise KeyError(f"Missing expected key in NPZ: {e}. Keys found: {list(d.keys())}")

    # 2. Load feature names if available, else default
    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    # 3. Strict Shape Assertions (Fail Fast)
    print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape}")

    assert X_train.ndim == 2, "X_train must be 2D"
    assert X_test.ndim == 2,  "X_test must be 2D"
    assert X_train.shape[0] == y_train.shape[0], f"Mismatch: X_train rows {X_train.shape[0]} != y_train {y_train.shape[0]}"
    # But for a standard ML setup, they must match.
    assert X_test.shape[0] == y_test.shape[0], f"Mismatch: X_test rows {X_test.shape[0]} != y_test {y_test.shape[0]}"
    assert len(feature_names) == X_train.shape[1], "Mismatch between feature_names length and X columns"

    return X_train, y_train, X_test, y_test, feature_names

# Load Data
X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]

# Ensure labels are a sorted list for consistent log_loss usage
all_labels = sorted(np.unique(y_train).tolist())
print(f"Labels detected: {all_labels}")

# Baseline: Mean of training data
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

print("\nTraining models...")
for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)

    # Ensure it is a classifier
    if not is_classifier(model):
         # If get_model returns a regressor but data implies classification, warn/error
         print(f"Warning: {m_name} does not seem to be a classifier (is_classifier=False).")

    model.fit(X_train, y_train)
    trained_models[m_name] = model

print("Models trained:", list(trained_models.keys()))

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true) # Ensure int
        self.all_labels = all_labels

        # Setup GT player interface with REAL feature names
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[feature_names[j] for j in feature_subset]
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        # Check if model supports probability
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            # Fallback for models like SVM (if probability=False) or others
            # Warning: log_loss is not strictly defined without probs,
            # but we can use 0-1 loss or raise error.
            # Here we try to interpret decision function or hard predict.
            pred = self.model.predict(X_in)[0]
            # Simple 0-1 Loss fallback
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        X_in = self._x_for_coalition(S)
        return -self._loss_value(X_in)

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("shapley_shubik", exact_shapley_shubik),
    ("banzhaf", exact_banzhaf_index),
]

records = []

print(f"\nRunning explanation on {len(test_idx)} samples using {len(METHODS)} methods...")

for model_name, model in trained_models.items():
    # Check if this model might fail log_loss (e.g. SVM without probability=True)
    if model_name == "svm" and not getattr(model, "probability", True) and not hasattr(model, "predict_proba"):
        print(f"Warning: SVM model might lack predict_proba. Fallback 0-1 loss will be used.")

    for i, idx in enumerate(test_idx):
        x = X_test[idx]
        y = y_test[idx]

        # Use ALL features (exact computation)
        feat_subset = np.arange(n_features)

        util = LocalLossUtility(
            model=model,
            x_full=x,
            y_true=y,
            baseline_full=baseline_full,
            feature_subset=feat_subset,
            feature_names=feature_names,
            all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                # Create named dictionary for clarity
                alloc_named = {feature_names[j]: float(alloc[j]) for j in range(n_features)}

                rec = {
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y),
                    "method": method_name,
                    "alloc_full": alloc.tolist(),
                    "alloc_named": alloc_named
                }
                records.append(rec)
            except Exception as e:
                print(f"Error in {method_name} (Model: {model_name}, Idx: {idx}): {e}")

df = pd.DataFrame(records)
print("\nExperiment Complete. First 5 rows:")
print(df.head())

out_file = os.path.join(OUT_DIR, "loss_utility_results.csv")
df.to_csv(out_file, index=False)
print(f"\nResults saved to: {out_file}")

Loading data from ../datasets/soil_spi_preprocessed.npz...
Shapes -> X_train: (4000, 7), y_train: (4000,), X_test: (1000, 7)
Labels detected: [0, 1, 2]

Training models...
Models trained: ['logistic', 'svm', 'random_forest']

Running explanation on 50 samples using 6 methods...

Experiment Complete. First 5 rows:
      model  test_index  y_true          method  \
0  logistic         210       0         shapley   
1  logistic         210       0      least_core   
2  logistic         210       0       nucleolus   
3  logistic         210       0          gately   
4  logistic         210       0  shapley_shubik   

                                          alloc_full  \
0  [0.4332037222255711, 0.7544623374476076, 0.590...   
1  [-0.01580790443075716, 0.00933958524815701, -0...   
2  [-0.014365911691893144, -0.0007680868699476388...   
3  [-0.027913659337746477, -0.004956302414021785,...   
4  [0.16666666666666666, 0.16666666666666666, 0.1...   

                                         

In [ ]:
import os
import sys
import warnings
import logging
import json
from pathlib import Path
from typing import Dict, List, Callable, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

# Setup Paths
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import GT methods
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.shapley_shubik import exact_shapley_shubik
from cgt_xai.concepts.banzhaf import exact_banzhaf_index
from cgt_xai.models import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_localsoil/loss_utility_results.csv"
OUT_DIR = Path("./output_soil/quantus_evaluation_soil")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()
    return X_train, X_test, y_train, y_test

# 2. perturbation & distance functions

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    """
    Safely perturbs tabular data by replacing with baseline.
    """
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))


def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,  # Faithfulness uses subset_size
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,   # Selectivity uses patch_size
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1, # Monotonicity uses features_in_step
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1, # PixelFlipping uses features_in_step
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),

        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=5, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=5, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    for k in top_k_list:
        if k > X_test_2d.shape[1]: continue
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]
        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k)

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "nucleolus": exact_nucleolus,
        "gately": exact_gately_value,
        "shapley_shubik": exact_shapley_shubik,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, 'item') else y_in

            util = LocalLossUtility(
                model=real_model, x_full=x, y_true=y,
                baseline_full=baseline_full, feature_subset=np.arange(len(x)),
                feature_names=feature_names, all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    X_train, X_test, y_train, y_test = load_data_strict()
    baseline = np.mean(X_train, axis=0)

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset = y_test[valid_indices]

    # Quantus Input: (N, 1, F)
    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    all_eval_rows = []
    metrics = get_metrics(baseline)

    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    logger.info(f"Loaded {len(valid_indices)} samples for evaluation.")

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()
        all_labels = sorted(np.unique(y_train).tolist())

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(X_train.shape[1]))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    if isinstance(val_str, str):
                        val_arr = np.array(json.loads(val_str))
                    else:
                        val_arr = np.array(val_str)
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

            # Global Metrics
            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_faith,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            # Aopc
            try:
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            # Robustness
            explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            all_eval_rows.append(row_stats)
            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

    df_eval = pd.DataFrame(all_eval_rows)
    out_path = OUT_DIR / "quantussoil_results_full_extended.csv"
    df_eval.to_csv(out_path, index=False)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()

            model          method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  local_lipschitz_estimate  max_sensitivity
0        logistic         shapley           0.821429               0.797687     0.203708          0.44        0.203708    0.482146    1.491382     0.442269                  0.430822         0.566439
1        logistic      least_core           0.810747               0.836378     0.207823          0.36        0.207823    0.458666    1.546588     0.299024                  0.554232         1.164727
2        logistic       nucleolus           0.888345               0.861762     0.214939          0.38        0.214939    0.424505    1.582988     0.287040                  0.569848         1.370556
3        logistic          gately           0.879901               0.894976     0.216727          0.38        0.216727    0.400242    1.626086     0.303459                  0.654425         1.451075
4    

In [ ]:
import os
import sys
import warnings
import logging
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

# Setup Paths
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import GT methods
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.shapley_shubik import exact_shapley_shubik
from cgt_xai.concepts.banzhaf import exact_banzhaf_index
from cgt_xai.models import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_localsoil/loss_utility_results.csv"
OUT_DIR = Path("./outputs_soil/quantus_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
# Soil dataset has 7 features
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        # Flatten (N, 1, F) -> (N, F) or (N, F) -> (N, F)
        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    return X_train, X_test, y_train, y_test, feature_names

# 2. perturbation

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    B = arr.shape[0]
    for b in range(B):
        idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
        if arr.ndim == 2:
            apply_mask(arr_pert[b], idx_list)
        elif arr.ndim == 3:
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 3. metrics

def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),

        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=5, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=3, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    # Ensure K doesn't exceed 7 (n_features)
    valid_k_list = [k for k in top_k_list if k <= X_test_2d.shape[1]]

    for k in valid_k_list:
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]
        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k) if drops_all_k else np.nan

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "nucleolus": exact_nucleolus,
        "gately": exact_gately_value,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, 'item') else y_in

            util = LocalLossUtility(
                model=real_model, x_full=x, y_true=y,
                baseline_full=baseline_full, feature_subset=np.arange(len(x)),
                feature_names=feature_names, all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    if not os.path.exists(RESULTS_CSV_PATH):
        print(f"Error: Results file not found at {RESULTS_CSV_PATH}. Run script 1 first.")
        return

    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset = y_test[valid_indices]

    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    all_eval_rows = []
    metrics = get_metrics(baseline)

    logger.info(f"Loaded {len(valid_indices)} samples for evaluation.")

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(X_train.shape[1]))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    try:
                        if isinstance(val_str, str):
                            val_arr = np.array(ast.literal_eval(val_str))
                        else:
                            val_arr = np.array(val_str)
                    except Exception:
                        val_arr = np.zeros(X_train.shape[1])
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_faith,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            try:
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            all_eval_rows.append(row_stats)
            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

    df_eval = pd.DataFrame(all_eval_rows)
    out_path = OUT_DIR / "quantus_results_soil.csv"
    df_eval.to_csv(out_path, index=False)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()

            model          method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  local_lipschitz_estimate  max_sensitivity
0        logistic         shapley           0.846266               0.797687     0.203708          0.44        0.203708    0.482146    1.491382     0.442269                  0.419959         0.574997
1        logistic      least_core           0.835995               0.836378     0.207823          0.36        0.207823    0.458666    1.546588     0.299024                  0.558155         1.266122
2        logistic       nucleolus           0.888639               0.861762     0.214939          0.38        0.214939    0.424505    1.582988     0.287040                  0.642714         1.268549
3        logistic          gately           0.926480               0.894976     0.216727          0.38        0.216727    0.400242    1.626086     0.303459                  1.035421         1.399692
4    

In [1]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import log_loss
from sklearn.base import is_classifier

DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
RESULTS_CSV = "./outputs_localsoil/loss_utility_results.csv"
PLOT_DIR = "./outputs_localsoil/insertion_deletion_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_STEPS = 20

import sys
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model

d = np.load(DATA_NPZ_PATH, allow_pickle=True)
X_train = d["X_train"].astype(float)
X_test  = d["X_test"].astype(float)
y_train = d["y_train"].astype(int).ravel()
y_test  = d["y_test"].astype(int).ravel()

if "feature_names" in d:
    feature_names = d["feature_names"].tolist()
else:
    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

n_features = X_train.shape[1]
baseline_full = np.mean(X_train, axis=0)
all_labels = sorted(np.unique(y_train).tolist())

MODEL_NAMES = ["logistic", "svm", "random_forest"]
trained_models = {}

for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

df = pd.read_csv(RESULTS_CSV)

# parse alloc_full safely
def parse_alloc(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)
    return np.array(x, dtype=float)

df["alloc_full_parsed"] = df["alloc_full"].apply(parse_alloc)

def compute_instance_loss(model, x_row, y_true):
    x_row = x_row.reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(x_row)
        return float(log_loss([y_true], proba, labels=all_labels))
    else:
        pred = model.predict(x_row)[0]
        return 0.0 if pred == y_true else 1.0

def deletion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = x.copy()
        remove_idx = order[:k]
        x_mod[remove_idx] = baseline[remove_idx]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def insertion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = baseline.copy()
        add_idx = order[:k]
        x_mod[add_idx] = x[add_idx]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def curve_auc(x, y):
    return float(np.trapz(y, x))

summary_rows = []

for model_name in sorted(df["model"].unique()):
    model = trained_models[model_name]

    for method_name in sorted(df["method"].unique()):
        sub = df[(df["model"] == model_name) & (df["method"] == method_name)].copy()

        deletion_curves = []
        insertion_curves = []

        for _, row in sub.iterrows():
            idx = int(row["test_index"])
            y_true = int(row["y_true"])
            attribution = row["alloc_full_parsed"]

            x = X_test[idx]

            x_del, y_del = deletion_curve_loss(
                model=model,
                x=x,
                y_true=y_true,
                attribution=attribution,
                baseline=baseline_full,
                n_steps=N_STEPS
            )

            x_ins, y_ins = insertion_curve_loss(
                model=model,
                x=x,
                y_true=y_true,
                attribution=attribution,
                baseline=baseline_full,
                n_steps=N_STEPS
            )

            deletion_curves.append(y_del)
            insertion_curves.append(y_ins)

        deletion_curves = np.array(deletion_curves)
        insertion_curves = np.array(insertion_curves)

        del_mean = deletion_curves.mean(axis=0)
        del_std = deletion_curves.std(axis=0)
        ins_mean = insertion_curves.mean(axis=0)
        ins_std = insertion_curves.std(axis=0)

        del_auc = curve_auc(x_del, del_mean)
        ins_auc = curve_auc(x_ins, ins_mean)

        summary_rows.append({
            "model": model_name,
            "method": method_name,
            "deletion_auc_loss": del_auc,
            "insertion_auc_loss": ins_auc,
        })

        plt.figure(figsize=(6, 4))
        plt.plot(x_del, del_mean, label=method_name)
        plt.fill_between(x_del, del_mean - del_std, del_mean + del_std, alpha=0.2)
        plt.xlabel("Fraction of features perturbed")
        plt.ylabel("Loss")
        plt.title(f"Deletion Curve (Loss)\nModel={model_name}, Method={method_name}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f"deletion_{model_name}_{method_name}.png"), dpi=200)
        plt.close()

        plt.figure(figsize=(6, 4))
        plt.plot(x_ins, ins_mean, label=method_name)
        plt.fill_between(x_ins, ins_mean - ins_std, ins_mean + ins_std, alpha=0.2)
        plt.xlabel("Fraction of features restored")
        plt.ylabel("Loss")
        plt.title(f"Insertion Curve (Loss)\nModel={model_name}, Method={method_name}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f"insertion_{model_name}_{method_name}.png"), dpi=200)
        plt.close()

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(PLOT_DIR, "insertion_deletion_auc_summary.csv"), index=False)

print(summary_df)
print(f"\nPlots and summary saved in: {PLOT_DIR}")

            model          method  deletion_auc_loss  insertion_auc_loss
0        logistic         banzhaf           1.586257            0.595537
1        logistic          gately           0.959939            0.791699
2        logistic      least_core           0.969515            0.835157
3        logistic       nucleolus           0.906330            0.791405
4        logistic         shapley           1.568606            0.597652
5        logistic  shapley_shubik           1.863144            0.621435
6   random_forest         banzhaf           2.055201            0.412556
7   random_forest          gately           1.989745            0.523828
8   random_forest      least_core           1.566335            0.530984
9   random_forest       nucleolus           1.941855            0.511782
10  random_forest         shapley           1.945183            0.651394
11  random_forest  shapley_shubik           3.573431            0.416832
12            svm         banzhaf           4.10183

In [3]:
import os
import ast
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import log_loss

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
RESULTS_CSV = "./outputs_localsoil/loss_utility_results.csv"
OUTPUT_DIR = "./outputs_localsoil/insertion_deletion_plots_combined"

RANDOM_STATE = 42
N_STEPS = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)

METHOD_NAME_MAP = {
    "shapley": "Shapley",
    "banzhaf": "Banzhaf",
    "gately": "Gately",
    "least_core": "Least Core",
    "nucleolus": "Nucleolus"
}

# Import project modules
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model

# Load data
data = np.load(DATA_NPZ_PATH, allow_pickle=True)
X_train = data["X_train"].astype(float)
X_test  = data["X_test"].astype(float)
y_train = data["y_train"].astype(int).ravel()
y_test  = data["y_test"].astype(int).ravel()

if "feature_names" in data:
    feature_names = data["feature_names"].tolist()
else:
    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

baseline_full = np.mean(X_train, axis=0)
all_labels = sorted(np.unique(y_train).tolist())

# Train models
MODEL_NAMES = sorted(pd.read_csv(RESULTS_CSV)["model"].unique().tolist())

trained_models = {}
for model_name in MODEL_NAMES:
    print(f"Training model: {model_name}")
    model = get_model(model_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[model_name] = model

# Load attributions
df = pd.read_csv(RESULTS_CSV)

def parse_alloc(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)
    return np.array(x, dtype=float)

df["alloc_full_parsed"] = df["alloc_full"].apply(parse_alloc)

# Loss function
def compute_instance_loss(model, x_row, y_true):
    x_row = x_row.reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(x_row)
        return float(log_loss([y_true], proba, labels=all_labels))
    else:
        pred = model.predict(x_row)[0]
        return 0.0 if pred == y_true else 1.0

# Curves
def deletion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = x.copy()
        x_mod[order[:k]] = baseline[order[:k]]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def insertion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = baseline.copy()
        x_mod[order[:k]] = x[order[:k]]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def curve_auc(x, y):
    return float(np.trapezoid(y, x))

# Main loop
summary_rows = []

for model_name in sorted(df["model"].unique()):
    print(f"\nProcessing model: {model_name}")
    model = trained_models[model_name]

    methods = sorted([m for m in df[df["model"] == model_name]["method"].unique()
    if m != "shapley_shubik"
])

    deletion_plot_data = {}
    insertion_plot_data = {}

    for method_name in methods:
        pretty_name = METHOD_NAME_MAP.get(method_name, method_name)

        sub = df[(df["model"] == model_name) & (df["method"] == method_name)]

        deletion_curves = []
        insertion_curves = []

        for _, row in sub.iterrows():
            idx = int(row["test_index"])
            y_true = int(row["y_true"])
            attribution = row["alloc_full_parsed"]

            x = X_test[idx]

            x_del, y_del = deletion_curve_loss(model, x, y_true, attribution, baseline_full, N_STEPS)
            x_ins, y_ins = insertion_curve_loss(model, x, y_true, attribution, baseline_full, N_STEPS)

            deletion_curves.append(y_del)
            insertion_curves.append(y_ins)

        deletion_curves = np.array(deletion_curves)
        insertion_curves = np.array(insertion_curves)

        del_mean = deletion_curves.mean(axis=0)
        del_std = deletion_curves.std(axis=0)

        # Normalize
        del_mean = (del_mean - del_mean[0]) / (del_mean[-1] - del_mean[0] + 1e-8)
        del_std = del_std / (del_mean[-1] - del_mean[0] + 1e-8)

        ins_mean = insertion_curves.mean(axis=0)
        ins_std = insertion_curves.std(axis=0)

        # Normalize
        ins_mean = (ins_mean - ins_mean[-1]) / (ins_mean[0] - ins_mean[-1] + 1e-8)
        ins_std = ins_std / (ins_mean[0] - ins_mean[-1] + 1e-8)

        del_auc = curve_auc(x_del, del_mean)
        ins_auc = curve_auc(x_ins, ins_mean)

        summary_rows.append({
            "model": model_name,
            "method": method_name,
            "deletion_auc_loss": del_auc,
            "insertion_auc_loss": ins_auc
        })

        deletion_plot_data[pretty_name] = (x_del, del_mean, del_std, del_auc)
        insertion_plot_data[pretty_name] = (x_ins, ins_mean, ins_std, ins_auc)

    plt.figure(figsize=(8, 5))
    for name, (x_vals, mean, std, auc_val) in deletion_plot_data.items():
        plt.plot(x_vals, mean, label=f"{name} (AUC={auc_val:.4f})")
        plt.fill_between(x_vals, mean - std, mean + std, alpha=0.15)

    plt.xlabel("Fraction of Features Perturbed")
    plt.ylabel("Loss")
    plt.title(f"Deletion Curves - {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/deletion_{model_name}.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 5))
    for name, (x_vals, mean, std, auc_val) in insertion_plot_data.items():
        plt.plot(x_vals, mean, label=f"{name} (AUC={auc_val:.4f})")
        plt.fill_between(x_vals, mean - std, mean + std, alpha=0.15)

    plt.xlabel("Fraction of Features Restored")
    plt.ylabel("Loss")
    plt.title(f"Insertion Curves - {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/insertion_{model_name}.png", dpi=300)
    plt.close()

# Save summary
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/auc_summary.csv", index=False)

print("\nDone!")
print(summary_df)

Training model: logistic
Training model: random_forest
Training model: svm

Processing model: logistic

Processing model: random_forest

Processing model: svm

Done!
            model      method  deletion_auc_loss  insertion_auc_loss
0        logistic     banzhaf           0.715042            0.170422
1        logistic      gately           0.370742            0.278257
2        logistic  least_core           0.376006            0.302147
3        logistic   nucleolus           0.341272            0.278095
4        logistic     shapley           0.705339            0.171585
5   random_forest     banzhaf           1.375730            0.160931
6   random_forest      gately           1.327323            0.243221
7   random_forest  least_core           1.014195            0.248513
8   random_forest   nucleolus           1.291907            0.234313
9   random_forest     shapley           1.294368            0.337561
10            svm     banzhaf           0.609773            0.134729
11    

In [7]:
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

def safe_spearman_abs(g, p):
    g_abs = np.abs(g)
    p_abs = np.abs(p)

    # Spearman undefined if either vector is constant
    if np.allclose(g_abs, g_abs[0]) or np.allclose(p_abs, p_abs[0]):
        return np.nan
    return spearmanr(g_abs, p_abs).correlation

def compute_metrics(gt_vecs, pred_vecs):
    spearman_list = []
    cosine_list = []
    mae_list = []
    rmse_list = []

    for g, p in zip(gt_vecs, pred_vecs):
        sp = safe_spearman_abs(g, p)
        cos = cosine_similarity(g.reshape(1, -1), p.reshape(1, -1))[0][0]
        mae = mean_absolute_error(g, p)
        rmse = np.sqrt(mean_squared_error(g, p))

        spearman_list.append(sp)
        cosine_list.append(cos)
        mae_list.append(mae)
        rmse_list.append(rmse)

    return {
        "spearman": np.nanmean(spearman_list),
        "cosine": np.mean(cosine_list),
        "mae": np.mean(mae_list),
        "rmse": np.mean(rmse_list),
        "num_valid_spearman": np.sum(~np.isnan(spearman_list)),
        "num_total": len(spearman_list),
    }


results = []

for (model, method), group in df.groupby(["model", "method"]):

    gt_vecs = []
    pred_vecs = []

    for _, row in group.iterrows():
        idx = int(row["test_index"])

        gt_vecs.append(GT[idx])
        pred_vecs.append(row["alloc_full"])

    metrics = compute_metrics(gt_vecs, pred_vecs)

    results.append({
        "model": model,
        "method": method,
        **metrics
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(["model", "spearman"], ascending=[True, False])

print("\n===== Ground Truth Comparison (Corrected) =====\n")
print(results_df.round(4))


===== Ground Truth Comparison (Corrected) =====

            model          method  spearman  cosine      mae     rmse  \
4        logistic         shapley    0.5200  0.3105  15.4761  22.1267   
0        logistic         banzhaf    0.4829  0.3056  17.5162  25.4830   
2        logistic      least_core    0.2636  0.1366  15.5394  22.2826   
3        logistic       nucleolus    0.2279  0.1358  15.5410  22.2883   
1        logistic          gately    0.2079  0.1792  15.5333  22.2758   
5        logistic  shapley_shubik    0.0199  0.4094  15.5271  22.2663   
6   random_forest         banzhaf    0.3171  0.2123  36.1308  51.0519   
10  random_forest         shapley    0.2836  0.1917  15.3693  21.9317   
9   random_forest       nucleolus    0.1300  0.0168  15.5160  22.2477   
11  random_forest  shapley_shubik    0.0997  0.3645  15.5299  22.2635   
8   random_forest      least_core    0.0607  0.0409  15.5049  22.2145   
7   random_forest          gately    0.0513  0.0534  15.4189  22.0821   
1

In [2]:
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.metrics import log_loss
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
OUT_DIR = "./outputs_localsoil_multisize"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

DATA_SIZES = [500, 1000]

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.shapley_shubik import exact_shapley_shubik
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test  = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test  = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")

    return X_train, y_train, X_test, y_test, feature_names

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[feature_names[j] for j in feature_subset]
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in):
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("shapley_shubik", exact_shapley_shubik),
    ("banzhaf", exact_banzhaf_index),
]

MODEL_NAMES = ["logistic", "svm", "random_forest"]

def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
    n_features = X_train_orig.shape[1]

    rng_test = np.random.default_rng(SAMPLE_SEED)
    test_idx = rng_test.choice(
        len(X_test),
        size=min(N_TEST_SAMPLES, len(X_test)),
        replace=False
    )
    print(f"Using {len(test_idx)} fixed test samples")

    for size in DATA_SIZES:
        print("\n" + "=" * 60)
        print(f"Running for dataset size: {size}")
        print("=" * 60)

        # Nested subsampling: 500 will be a subset of 1000
        rng_sub = np.random.default_rng(SAMPLE_SEED)
        indices = rng_sub.choice(len(X_train_orig), size=size, replace=False)

        X_train = X_train_orig[indices]
        y_train = y_train_orig[indices]
        size_tag = str(size)

        print(f"Train shape: {X_train.shape}")

        baseline_full = np.mean(X_train, axis=0)
        all_labels = sorted(np.unique(y_train).tolist())

        # Train models
        trained_models = {}
        print("Training models...")
        for m_name in MODEL_NAMES:
            model = get_model(m_name, random_state=RANDOM_STATE)
            model.fit(X_train, y_train)
            trained_models[m_name] = model

        # Attribution
        records = []

        for model_name, model in trained_models.items():
            for idx in test_idx:
                x = X_test[idx]
                y = y_test[idx]

                util = LocalLossUtility(
                    model=model,
                    x_full=x,
                    y_true=y,
                    baseline_full=baseline_full,
                    feature_subset=np.arange(n_features),
                    feature_names=feature_names,
                    all_labels=all_labels
                )

                for method_name, method_fn in METHODS:
                    try:
                        res = method_fn(util)
                        alloc = getattr(res, "values", res)
                        alloc = np.asarray(alloc, dtype=float).ravel()

                        records.append({
                            "dataset_size": size_tag,
                            "model": model_name,
                            "test_index": int(idx),
                            "y_true": int(y),
                            "method": method_name,
                            "alloc_full": json.dumps(alloc.tolist())
                        })
                    except Exception as e:
                        print(f"Error: {method_name}, model={model_name}, idx={idx}: {e}")

        df = pd.DataFrame(records)
        out_file = os.path.join(OUT_DIR, f"results_size_{size_tag}.csv")
        df.to_csv(out_file, index=False)
        print(f"Saved: {out_file}")

if __name__ == "__main__":
    main()

Original train size: 4000
Test size: 1000
Using 50 fixed test samples

Running for dataset size: 500
Train shape: (500, 7)
Training models...
Saved: ./outputs_localsoil_multisize/results_size_500.csv

Running for dataset size: 1000
Train shape: (1000, 7)
Training models...
Saved: ./outputs_localsoil_multisize/results_size_1000.csv


In [6]:
import os
import sys
import warnings
import logging
import json
from pathlib import Path
from typing import Dict, List
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

# Setup Paths
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import GT methods
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.shapley_shubik import exact_shapley_shubik
from cgt_xai.concepts.banzhaf import exact_banzhaf_index
from cgt_xai.models import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"

# Existing full-data attribution results from original script
FULL_RESULTS_CSV_PATH = "./outputs_localsoil/loss_utility_results.csv"

MULTISIZE_RESULTS_DIR = "./outputs_localsoil_multisize"

OUT_DIR = Path("./output_soil/quantus_evaluation_soil_multisize")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

# Evaluate these dataset sizes
DATA_SIZES = ["500", "1000", "full"]

# Must match Script 1 / original script
N_TEST_SAMPLES = 50
SAMPLE_SEED = 123

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1:
                out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    return X_train, X_test, y_train, y_test, feature_names

def get_train_subset(X_train_orig, y_train_orig, size_tag):
    """
    Returns the exact training subset corresponding to Script 1 / original run.
    """
    if size_tag == "full":
        return X_train_orig, y_train_orig

    size = int(size_tag)
    rng_sub = np.random.default_rng(SAMPLE_SEED)
    indices = rng_sub.choice(len(X_train_orig), size=size, replace=False)
    X_train = X_train_orig[indices]
    y_train = y_train_orig[indices]
    return X_train, y_train

def get_results_csv_path(size_tag):
    if size_tag == "full":
        return FULL_RESULTS_CSV_PATH
    return os.path.join(MULTISIZE_RESULTS_DIR, f"results_size_{size_tag}.csv")

# 2. perturbation & distance functions

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get("lower_bound", 0.01)
    upper = kwargs.get("upper_bound", 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 3. metric definitions

def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=3,
            lower_bound=0.1,
            upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    for k in top_k_list:
        if k > X_test_2d.shape[1]:
            continue

        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]

        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))

    return np.mean(drops_all_k) if len(drops_all_k) > 0 else np.nan

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "gately": exact_gately_value,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, "item") else y_in

            util = LocalLossUtility(
                model=real_model,
                x_full=x,
                y_true=y,
                baseline_full=baseline_full,
                feature_subset=np.arange(len(x)),
                feature_names=feature_names,
                all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, "values", res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    X_train_orig, X_test, y_train_orig, y_test, feature_names = load_data_strict()

    all_eval_rows = []

    for size_tag in DATA_SIZES:
        logger.info("=" * 80)
        logger.info(f"Evaluating dataset size: {size_tag}")
        logger.info("=" * 80)

        results_csv_path = get_results_csv_path(size_tag)
        if not os.path.exists(results_csv_path):
            logger.warning(f"Missing attribution file: {results_csv_path}")
            continue

        df_results = pd.read_csv(results_csv_path)
        VALID_METHODS = ["shapley", "least_core", "gately", "banzhaf"]
        df_results = df_results[df_results["method"].isin(VALID_METHODS)]

        # Reconstruct the exact training data used for this size
        X_train, y_train = get_train_subset(X_train_orig, y_train_orig, size_tag)

        baseline = np.mean(X_train, axis=0)
        metrics = get_metrics(baseline)
        all_labels = sorted(np.unique(y_train).tolist())

        # Use the exact test indices saved implicitly by common seed/setup
        valid_indices = sorted(df_results["test_index"].unique())
        X_subset = X_test[valid_indices]
        y_subset = y_test[valid_indices]

        # Quantus input shape: (N, 1, F)
        x_batch = X_subset[:, np.newaxis, :]

        models_to_run = df_results["model"].unique()
        methods_to_run = df_results["method"].unique()

        logger.info(f"Loaded {len(valid_indices)} test samples for size {size_tag}.")

        for model_name in models_to_run:
            logger.info(f"--- Evaluating Model: {model_name} ---")

            sk_model = get_model(model_name, random_state=RANDOM_STATE)
            sk_model.fit(X_train, y_train)
            torch_model = SklearnTorchWrapper(sk_model).eval()

            df_model = df_results[df_results["model"] == model_name]

            for method_name in methods_to_run:
                logger.info(f"   Method: {method_name}")

                attributions_list = []
                for idx in valid_indices:
                    row = df_model[
                        (df_model["test_index"] == idx) &
                        (df_model["method"] == method_name)
                    ]

                    if row.empty:
                        attributions_list.append(np.zeros(X_train.shape[1]))
                    else:
                        val_str = row.iloc[0]["alloc_full"]
                        if isinstance(val_str, str):
                            val_arr = np.array(json.loads(val_str))
                        else:
                            val_arr = np.array(val_str)
                        attributions_list.append(val_arr)

                a_batch_2d = np.array(attributions_list)
                a_batch_3d = a_batch_2d[:, np.newaxis, :]

                row_stats = {
                    "dataset_size": size_tag,
                    "dataset_train_n": int(X_train.shape[0]),
                    "model": model_name,
                    "method": method_name,
                }

                y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

                # Global metrics
                for m_name, metric_fn in metrics.items():
                    if m_name in ["max_sensitivity"]:
                        continue

                    try:
                        scores = metric_fn(
                            model=torch_model,
                            x_batch=x_batch,
                            y_batch=y_faith,
                            a_batch=a_batch_3d,
                            channel_first=True,
                            softmax=False
                        )
                        row_stats[m_name] = np.nanmean(scores)
                    except Exception as e:
                        logger.warning(f"     {m_name} Error: {e}")
                        row_stats[m_name] = np.nan

                # Aopc
                try:
                    aopc = compute_aopc_manual(
                        sk_model=sk_model,
                        X_test_2d=X_subset,
                        y_test=y_subset,
                        attributions_2d=a_batch_2d,
                        top_k_list=AOPC_TOPK_LIST,
                        baseline=baseline
                    )
                    row_stats["aopc_manual"] = aopc
                except Exception as e:
                    logger.warning(f"     aopc_manual Error: {e}")
                    row_stats["aopc_manual"] = np.nan

                # Robustness
                explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

                for m_name in ["max_sensitivity"]:
                    try:
                        scores = metrics[m_name](
                            model=torch_model,
                            x_batch=x_batch,
                            y_batch=y_subset,
                            explain_func=explain_func,
                            channel_first=True,
                            softmax=False
                        )
                        row_stats[m_name] = np.nanmean(scores)
                    except Exception as e:
                        logger.warning(f"     {m_name} Error: {e}")
                        row_stats[m_name] = np.nan

                all_eval_rows.append(row_stats)

                logger.info(
                    f"     -> Faith_Corr: {row_stats.get('faithfulness_corr', np.nan):.4f}, "
                    f"MaxSens: {row_stats.get('max_sensitivity', np.nan):.4f}"
                )

    df_eval = pd.DataFrame(all_eval_rows)

    out_csv = OUT_DIR / "quantussoil_results_multisize.csv"
    df_eval.to_csv(out_csv, index=False)

    print("\nSaved Quantus results to:")
    print(out_csv)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()


Saved Quantus results to:
output_soil/quantus_evaluation_soil_multisize/quantussoil_results_multisize.csv
   dataset_size  dataset_train_n          model      method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  max_sensitivity
0           500              500       logistic     shapley           0.839848               0.800932     0.217976          0.40        0.217976    0.480019    1.498706     0.415863         0.520705
1           500              500       logistic  least_core           0.861242               0.838164     0.216668          0.36        0.216668    0.434851    1.584812     0.274879         0.830249
2           500              500       logistic      gately           0.927695               0.904967     0.221890          0.34        0.221890    0.386968    1.654221     0.270046         1.020277
3           500              500       logistic     banzhaf           0.828377               0.79

In [1]:
#Global Experimnet
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.metrics import log_loss, accuracy_score
from dataclasses import dataclass
from typing import List

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
OUT_DIR = "./outputs_globalsoil_multisize"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SEED = 123
N_GLOBAL_SAMPLES = 50

DATA_SIZES = [500, 1000, "full"]

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

# Data structure
@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]

    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

# Global loss utility
class GlobalLossUtility:
    def __init__(self, model, X_eval, y_eval, baseline_full, feature_names, all_labels):
        self.model = model
        self.X_eval = np.asarray(X_eval, dtype=float)
        self.y_eval = np.asarray(y_eval, dtype=int).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_names = feature_names
        self.all_labels = all_labels

        self.n_features = self.X_eval.shape[1]

        self.data = FeaturePlayers(
            indices=list(range(self.n_features)),
            data_names=[feature_names[j] for j in range(self.n_features)]
        )

    def _X_for_coalition(self, S):
        X_masked = np.tile(self.baseline_full, (self.X_eval.shape[0], 1))

        for j in S:
            j = int(j)
            X_masked[:, j] = self.X_eval[:, j]

        return X_masked

    def _loss_value(self, X_in):
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss(self.y_eval, proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)
            return float(np.mean(pred != self.y_eval))

    def __call__(self, S):
        return -self._loss_value(self._X_for_coalition(S))

# Methods / models
METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

MODEL_NAMES = ["logistic", "svm", "random_forest"]

# Helpers
def load_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")

    return X_train, y_train, X_test, y_test, feature_names

def get_train_subset(X_train_orig, y_train_orig, size_tag):
    if size_tag == "full":
        return X_train_orig, y_train_orig

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X_train_orig), size=int(size_tag), replace=False)

    return X_train_orig[idx], y_train_orig[idx]

# Main
def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_data(DATA_NPZ_PATH)

    rng_eval = np.random.default_rng(SAMPLE_SEED)
    eval_idx = rng_eval.choice(
        len(X_test),
        size=min(N_GLOBAL_SAMPLES, len(X_test)),
        replace=False
    )

    X_eval = X_test[eval_idx]
    y_eval = y_test[eval_idx]

    print(f"Using {len(eval_idx)} fixed global evaluation samples")

    for size_tag in DATA_SIZES:
        print("\n" + "=" * 80)
        print(f"GLOBAL ATTRIBUTION | Train size = {size_tag}")
        print("=" * 80)

        X_train, y_train = get_train_subset(X_train_orig, y_train_orig, size_tag)

        baseline_full = np.mean(X_train, axis=0)
        all_labels = sorted(np.unique(y_train).tolist())

        records = []

        for model_name in MODEL_NAMES:
            print(f"\nTraining model: {model_name}")

            model = get_model(model_name, random_state=RANDOM_STATE)
            model.fit(X_train, y_train)

            train_acc = accuracy_score(y_train, model.predict(X_train))
            test_acc = accuracy_score(y_test, model.predict(X_test))
            eval_acc = accuracy_score(y_eval, model.predict(X_eval))

            print(
                f"{model_name} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Full Test Acc: {test_acc:.4f} | "
                f"Eval Acc: {eval_acc:.4f}"
            )

            util = GlobalLossUtility(
                model=model,
                X_eval=X_eval,
                y_eval=y_eval,
                baseline_full=baseline_full,
                feature_names=feature_names,
                all_labels=all_labels
            )

            for method_name, method_fn in METHODS:
                print(f"  Computing {method_name}")

                try:
                    res = method_fn(util)
                    alloc = getattr(res, "values", res)
                    alloc = np.asarray(alloc, dtype=float).ravel()

                    records.append({
                        "setup": "global",
                        "dataset_size": str(size_tag),
                        "dataset_train_n": int(X_train.shape[0]),
                        "model": model_name,
                        "method": method_name,
                        "train_accuracy": float(train_acc),
                        "test_accuracy": float(test_acc),
                        "eval_accuracy": float(eval_acc),
                        "alloc_global": json.dumps(alloc.tolist()),
                        "eval_indices": json.dumps(eval_idx.tolist()),
                        "n_eval_samples": int(len(eval_idx))
                    })

                except Exception as e:
                    print(f"  ERROR: {method_name}, model={model_name}: {e}")

        df = pd.DataFrame(records)

        out_file = os.path.join(
            OUT_DIR,
            f"global_results_size_{size_tag}.csv"
        )

        df.to_csv(out_file, index=False)

        print(f"\nSaved: {out_file}")
        print(df[[
            "dataset_size",
            "dataset_train_n",
            "model",
            "method",
            "train_accuracy",
            "test_accuracy",
            "eval_accuracy"
        ]])

if __name__ == "__main__":
    main()

Original train size: 4000
Test size: 1000
Number of features: 7
Using 50 fixed global evaluation samples

GLOBAL ATTRIBUTION | Train size = 500

Training model: logistic
logistic | Train Acc: 0.8440 | Full Test Acc: 0.8420 | Eval Acc: 0.8400
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Training model: svm
svm | Train Acc: 0.9180 | Full Test Acc: 0.8760 | Eval Acc: 0.9200
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Training model: random_forest
random_forest | Train Acc: 1.0000 | Full Test Acc: 0.8670 | Eval Acc: 0.8800
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Saved: ./outputs_globalsoil_multisize/global_results_size_500.csv
   dataset_size  dataset_train_n          model      method  train_accuracy  \
0           500              500       logistic     shapley           0.844   
1           500          

In [2]:
import os
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

from sklearn.metrics import accuracy_score

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from cgt_xai.models import get_model

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"

GLOBAL_RESULTS_DIR = "./outputs_globalsoil_multisize"

OUT_DIR = Path("./output_soil/quantus_evaluation_soil_global_multisize")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SEED = 123

DATA_SIZES = ["500", "1000", "full"]
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

# Wrapper
class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1:
                out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

# Data
def load_data():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")

    return X_train, y_train, X_test, y_test, feature_names

def get_train_subset(X_train_orig, y_train_orig, size_tag):
    if size_tag == "full":
        return X_train_orig, y_train_orig

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X_train_orig), size=int(size_tag), replace=False)

    return X_train_orig[idx], y_train_orig[idx]

def get_global_csv(size_tag):
    return os.path.join(
        GLOBAL_RESULTS_DIR,
        f"global_results_size_{size_tag}.csv"
    )

# Perturbation functions
def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        for b in range(arr.shape[0]):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        for b in range(arr.shape[0]):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

# Aopc
def compute_aopc_manual(
    sk_model,
    X_test_2d,
    y_test,
    attributions_2d,
    top_k_list,
    baseline
):
    if not hasattr(sk_model, "predict_proba"):
        return np.nan

    base_probs = sk_model.predict_proba(X_test_2d)
    base_scores = base_probs[np.arange(len(y_test)), y_test]

    drops = []

    for k in top_k_list:
        if k > X_test_2d.shape[1]:
            continue

        X_perturbed = X_test_2d.copy()

        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]

        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]

        drops.append(np.mean(base_scores - new_scores))

    return np.mean(drops) if len(drops) > 0 else np.nan

# Metrics
def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(
            return_aggregate=True
        ),

        "complexity": quantus.Complexity(
            return_aggregate=True
        ),
    }

    return metrics

# Main
def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_data()

    all_rows = []

    for size_tag in DATA_SIZES:
        logger.info("=" * 80)
        logger.info(f"Evaluating GLOBAL results for train size: {size_tag}")
        logger.info("=" * 80)

        csv_path = get_global_csv(size_tag)

        if not os.path.exists(csv_path):
            logger.warning(f"Missing global result file: {csv_path}")
            continue

        df_global = pd.read_csv(csv_path)

        X_train, y_train = get_train_subset(
            X_train_orig,
            y_train_orig,
            size_tag
        )

        baseline = np.mean(X_train, axis=0)
        metrics = get_metrics(baseline)

        eval_indices = json.loads(df_global.iloc[0]["eval_indices"])

        X_eval = X_test[eval_indices]
        y_eval = y_test[eval_indices]

        x_batch = X_eval[:, np.newaxis, :]

        for model_name in df_global["model"].unique():
            logger.info(f"Model: {model_name}")

            sk_model = get_model(model_name, random_state=RANDOM_STATE)
            sk_model.fit(X_train, y_train)

            train_acc = accuracy_score(y_train, sk_model.predict(X_train))
            test_acc = accuracy_score(y_test, sk_model.predict(X_test))
            eval_acc = accuracy_score(y_eval, sk_model.predict(X_eval))

            logger.info(
                f"{model_name} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Full Test Acc: {test_acc:.4f} | "
                f"Eval Acc: {eval_acc:.4f}"
            )

            torch_model = SklearnTorchWrapper(sk_model).eval()

            df_model = df_global[df_global["model"] == model_name]

            for method_name in df_model["method"].unique():
                logger.info(f"  Method: {method_name}")

                row = df_model[df_model["method"] == method_name].iloc[0]

                global_attr = np.array(
                    json.loads(row["alloc_global"]),
                    dtype=float
                )

                # Same global attribution vector is applied to every sample.
                a_batch_2d = np.tile(global_attr, (len(X_eval), 1))
                a_batch_3d = a_batch_2d[:, np.newaxis, :]

                result = {
                    "setup": "global",
                    "dataset_size": size_tag,
                    "dataset_train_n": int(X_train.shape[0]),
                    "n_eval_samples": int(len(X_eval)),
                    "model": model_name,
                    "method": method_name,
                    "train_accuracy": float(train_acc),
                    "test_accuracy": float(test_acc),
                    "eval_accuracy": float(eval_acc),
                }

                for metric_name, metric_fn in metrics.items():
                    try:
                        scores = metric_fn(
                            model=torch_model,
                            x_batch=x_batch,
                            y_batch=y_eval,
                            a_batch=a_batch_3d,
                            channel_first=True,
                            softmax=False
                        )

                        result[metric_name] = float(np.nanmean(scores))

                    except Exception as e:
                        logger.warning(f"    {metric_name} error: {e}")
                        result[metric_name] = np.nan

                try:
                    result["aopc_manual"] = compute_aopc_manual(
                        sk_model=sk_model,
                        X_test_2d=X_eval,
                        y_test=y_eval,
                        attributions_2d=a_batch_2d,
                        top_k_list=AOPC_TOPK_LIST,
                        baseline=baseline
                    )

                except Exception as e:
                    logger.warning(f"    AOPC error: {e}")
                    result["aopc_manual"] = np.nan

                all_rows.append(result)

                logger.info(
                    f"    FaithCorr={result.get('faithfulness_corr', np.nan):.4f}, "
                    f"AOPC={result.get('aopc_manual', np.nan):.4f}"
                )

    df_out = pd.DataFrame(all_rows)

    out_csv = OUT_DIR / "quantussoil_results_global_multisize.csv"
    df_out.to_csv(out_csv, index=False)

    print("\nSaved global Quantus results to:")
    print(out_csv)

    print("\nFinal results:")
    print(df_out.to_string(index=False))

if __name__ == "__main__":
    main()

Original train size: 4000
Test size: 1000
Number of features: 7



Saved global Quantus results to:
output_soil/quantus_evaluation_soil_global_multisize/quantussoil_results_global_multisize.csv

Final results:
 setup dataset_size  dataset_train_n  n_eval_samples         model     method  train_accuracy  test_accuracy  eval_accuracy  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual
global          500              500              50      logistic    shapley         0.84400          0.842           0.84           0.465238               0.419956     0.421809          0.42        0.421809    0.431132    1.603119     0.363846
global          500              500              50      logistic least_core         0.84400          0.842           0.84           0.468686               0.422866     0.421809          0.00        0.421809    0.150987    1.908351     0.205566
global          500              500              50      logistic  nucleolus         0.84400          0.842          

In [12]:
import pandas as pd
import numpy as np
import json
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
from numpy.linalg import norm

# Ground Truth
GT = np.array([0.064, 0.401, 0.047, 0.138, 0.138, 0.060, 0.153])

# Normalize GT
GT = np.abs(GT)
GT = GT / (np.sum(GT) + 1e-8)

# Load Global Results
df = pd.read_csv("./outputs_globalsoil_multisize/global_results_size_500.csv")

results = []

print("\nGLOBAL vs GROUND TRUTH\n")

for _, row in df.iterrows():

    phi = np.array(json.loads(row["alloc_global"]))

    # Normalize attribution
    phi = np.abs(phi)
    phi = phi / (np.sum(phi) + 1e-8)

    # Metrics
    spearman = spearmanr(phi, GT).correlation
    rmse = np.sqrt(mean_squared_error(GT, phi))
    cosine = np.dot(phi, GT) / (norm(phi) * norm(GT) + 1e-8)

    results.append({
        "model": row["model"],
        "method": row["method"],
        "spearman": spearman,
        "cosine": cosine,
        "rmse": rmse
    })

    print(
        f"{row['model']:15} {row['method']:12} | "
        f"Spearman: {spearman:.4f} | "
        f"Cosine: {cosine:.4f} | "
        f"RMSE: {rmse:.4f}"
    )

# Save results
df_out = pd.DataFrame(results)
df_out.to_csv("global_gt_comparison_500.csv", index=False)

print("\nSaved to global_gt_comparison_500.csv")


GLOBAL vs GROUND TRUTH

RandomForestClassifier(n_estimators=200, random_state=42) shapley      | Spearman: 0.8469 | Cosine: 0.9548 | RMSE: 0.0611
RandomForestClassifier(n_estimators=200, random_state=42) least_core   | Spearman: -0.1261 | Cosine: 0.8104 | RMSE: 0.1071
RandomForestClassifier(n_estimators=200, random_state=42) nucleolus    | Spearman: -0.5225 | Cosine: 0.7236 | RMSE: 0.1274
RandomForestClassifier(n_estimators=200, random_state=42) gately       | Spearman: -0.3424 | Cosine: 0.7468 | RMSE: 0.1225
RandomForestClassifier(n_estimators=200, random_state=42) banzhaf      | Spearman: 0.8829 | Cosine: 0.9448 | RMSE: 0.0709

Saved to global_gt_comparison_500.csv


In [4]:
import pandas as pd
import numpy as np
import json
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
from numpy.linalg import norm

# Ground Truth
GT = np.array([0.064, 0.401, 0.047, 0.138, 0.138, 0.060, 0.153])

# Normalize GT
GT = np.abs(GT)
GT = GT / (np.sum(GT) + 1e-8)

# Load Global Results
df = pd.read_csv("./outputs_globalsoil_multisize/global_results_size_1000.csv")

results = []

print("\nGLOBAL vs GROUND TRUTH\n")

for _, row in df.iterrows():

    phi = np.array(json.loads(row["alloc_global"]))

    # Normalize attribution
    phi = np.abs(phi)
    phi = phi / (np.sum(phi) + 1e-8)

    # Metrics
    spearman = spearmanr(phi, GT).correlation
    rmse = np.sqrt(mean_squared_error(GT, phi))
    cosine = np.dot(phi, GT) / (norm(phi) * norm(GT) + 1e-8)

    results.append({
        "model": row["model"],
        "method": row["method"],
        "spearman": spearman,
        "cosine": cosine,
        "rmse": rmse
    })

    print(
        f"{row['model']:15} {row['method']:12} | "
        f"Spearman: {spearman:.4f} | "
        f"Cosine: {cosine:.4f} | "
        f"RMSE: {rmse:.4f}"
    )

# Save results
df_out = pd.DataFrame(results)
df_out.to_csv("global_gt_comparison_500.csv", index=False)

print("\nSaved to global_gt_comparison_500.csv")


GLOBAL vs GROUND TRUTH

logistic        shapley      | Spearman: 0.7388 | Cosine: 0.9119 | RMSE: 0.0766
logistic        least_core   | Spearman: 0.0180 | Cosine: 0.7923 | RMSE: 0.1111
logistic        nucleolus    | Spearman: 0.3063 | Cosine: 0.8096 | RMSE: 0.1068
logistic        gately       | Spearman: 0.1982 | Cosine: 0.8038 | RMSE: 0.1082
logistic        banzhaf      | Spearman: 0.7388 | Cosine: 0.9128 | RMSE: 0.0765
svm             shapley      | Spearman: 0.8829 | Cosine: 0.9233 | RMSE: 0.0711
svm             least_core   | Spearman: 0.0721 | Cosine: 0.8496 | RMSE: 0.0962
svm             nucleolus    | Spearman: 0.5045 | Cosine: 0.8076 | RMSE: 0.1074
svm             gately       | Spearman: 0.0180 | Cosine: 0.8198 | RMSE: 0.1044
svm             banzhaf      | Spearman: 0.7748 | Cosine: 0.9172 | RMSE: 0.0744
random_forest   shapley      | Spearman: 0.5406 | Cosine: 0.9481 | RMSE: 0.0747
random_forest   least_core   | Spearman: 0.2162 | Cosine: 0.9230 | RMSE: 0.0702
random_forest  

In [5]:
import pandas as pd
import numpy as np
import json
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
from numpy.linalg import norm

# Ground Truth
GT = np.array([0.064, 0.401, 0.047, 0.138, 0.138, 0.060, 0.153])

# Normalize GT
GT = np.abs(GT)
GT = GT / (np.sum(GT) + 1e-8)

# Load Global Results
df = pd.read_csv("./outputs_globalsoil_multisize/global_results_size_full.csv")

results = []

print("\nGLOBAL vs GROUND TRUTH\n")

for _, row in df.iterrows():

    phi = np.array(json.loads(row["alloc_global"]))

    # Normalize attribution
    phi = np.abs(phi)
    phi = phi / (np.sum(phi) + 1e-8)

    # Metrics
    spearman = spearmanr(phi, GT).correlation
    rmse = np.sqrt(mean_squared_error(GT, phi))
    cosine = np.dot(phi, GT) / (norm(phi) * norm(GT) + 1e-8)

    results.append({
        "model": row["model"],
        "method": row["method"],
        "spearman": spearman,
        "cosine": cosine,
        "rmse": rmse
    })

    print(
        f"{row['model']:15} {row['method']:12} | "
        f"Spearman: {spearman:.4f} | "
        f"Cosine: {cosine:.4f} | "
        f"RMSE: {rmse:.4f}"
    )

# Save results
df_out = pd.DataFrame(results)
df_out.to_csv("global_gt_comparison_500.csv", index=False)

print("\nSaved to global_gt_comparison_500.csv")


GLOBAL vs GROUND TRUTH

logistic        shapley      | Spearman: 0.7388 | Cosine: 0.9227 | RMSE: 0.0708
logistic        least_core   | Spearman: 0.0180 | Cosine: 0.8200 | RMSE: 0.1042
logistic        nucleolus    | Spearman: 0.3964 | Cosine: 0.8339 | RMSE: 0.1004
logistic        gately       | Spearman: 0.2523 | Cosine: 0.8279 | RMSE: 0.1021
logistic        banzhaf      | Spearman: 0.7388 | Cosine: 0.9244 | RMSE: 0.0702
svm             shapley      | Spearman: 0.7388 | Cosine: 0.9127 | RMSE: 0.0745
svm             least_core   | Spearman: 0.1081 | Cosine: 0.8205 | RMSE: 0.1045
svm             nucleolus    | Spearman: 0.5586 | Cosine: 0.8538 | RMSE: 0.0950
svm             gately       | Spearman: 0.1441 | Cosine: 0.8051 | RMSE: 0.1079
svm             banzhaf      | Spearman: 0.6847 | Cosine: 0.8949 | RMSE: 0.0835
random_forest   shapley      | Spearman: 0.5045 | Cosine: 0.9305 | RMSE: 0.0821
random_forest   least_core   | Spearman: 0.3424 | Cosine: 0.9251 | RMSE: 0.0691
random_forest  

In [7]:
#Global Experimnet
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.metrics import log_loss, accuracy_score
from dataclasses import dataclass
from typing import List

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
OUT_DIR = "./outputs_globalsoil_multisize"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SEED = 123
N_GLOBAL_SAMPLES = 50

DATA_SIZES = [500, 1000, "full"]

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

# Data structure
@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]

    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

# Global loss utility
class GlobalLossUtility:
    def __init__(self, model, X_eval, y_eval, baseline_full, feature_names, all_labels):
        self.model = model
        self.X_eval = np.asarray(X_eval, dtype=float)
        self.y_eval = np.asarray(y_eval, dtype=int).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_names = feature_names
        self.all_labels = all_labels

        self.n_features = self.X_eval.shape[1]

        self.data = FeaturePlayers(
            indices=list(range(self.n_features)),
            data_names=[feature_names[j] for j in range(self.n_features)]
        )

    def _X_for_coalition(self, S):
        X_masked = np.tile(self.baseline_full, (self.X_eval.shape[0], 1))

        for j in S:
            j = int(j)
            X_masked[:, j] = self.X_eval[:, j]

        return X_masked

    def _loss_value(self, X_in):
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss(self.y_eval, proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)
            return float(np.mean(pred != self.y_eval))

    def __call__(self, S):
        return -self._loss_value(self._X_for_coalition(S))

# Methods / models
METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

MODEL_NAMES = ["logistic", "svm", "random_forest"]

# Helpers
def load_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")

    return X_train, y_train, X_test, y_test, feature_names

def get_train_subset(X_train_orig, y_train_orig, size_tag):
    if size_tag == "full":
        return X_train_orig, y_train_orig

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X_train_orig), size=int(size_tag), replace=False)

    return X_train_orig[idx], y_train_orig[idx]

# Main
def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_data(DATA_NPZ_PATH)

    rng_eval = np.random.default_rng(SAMPLE_SEED)
    eval_idx = rng_eval.choice(
        len(X_test),
        size=min(N_GLOBAL_SAMPLES, len(X_test)),
        replace=False
    )

    X_eval = X_test[eval_idx]
    y_eval = y_test[eval_idx]

    print(f"Using {len(eval_idx)} fixed global evaluation samples")

    for size_tag in DATA_SIZES:
        print("\n" + "=" * 80)
        print(f"GLOBAL ATTRIBUTION | Train size = {size_tag}")
        print("=" * 80)

        X_train, y_train = get_train_subset(X_train_orig, y_train_orig, size_tag)

        baseline_full = np.mean(X_train, axis=0)
        all_labels = sorted(np.unique(y_train).tolist())

        records = []

        from sklearn.ensemble import RandomForestClassifier

        model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
        )
        model.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))
        eval_acc = accuracy_score(y_eval, model.predict(X_eval))

        print(
                f"{model} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Full Test Acc: {test_acc:.4f} | "
                f"Eval Acc: {eval_acc:.4f}"
            )

        util = GlobalLossUtility(
                model=model,
                X_eval=X_eval,
                y_eval=y_eval,
                baseline_full=baseline_full,
                feature_names=feature_names,
                all_labels=all_labels
            )

        for method_name, method_fn in METHODS:
                print(f"  Computing {method_name}")

                try:
                    res = method_fn(util)
                    alloc = getattr(res, "values", res)
                    alloc = np.asarray(alloc, dtype=float).ravel()

                    records.append({
                        "setup": "global",
                        "dataset_size": str(size_tag),
                        "dataset_train_n": int(X_train.shape[0]),
                        "model": model,
                        "method": method_name,
                        "train_accuracy": float(train_acc),
                        "test_accuracy": float(test_acc),
                        "eval_accuracy": float(eval_acc),
                        "alloc_global": json.dumps(alloc.tolist()),
                        "eval_indices": json.dumps(eval_idx.tolist()),
                        "n_eval_samples": int(len(eval_idx))
                    })

                except Exception as e:
                    print(f"  ERROR: {method_name}, model={model_name}: {e}")

        df = pd.DataFrame(records)

        out_file = os.path.join(
            OUT_DIR,
            f"global_results_size_{size_tag}.csv"
        )

        df.to_csv(out_file, index=False)

        print(f"\nSaved: {out_file}")
        print(df[[
            "dataset_size",
            "dataset_train_n",
            "model",
            "method",
            "train_accuracy",
            "test_accuracy",
            "eval_accuracy"
        ]])

if __name__ == "__main__":
    main()

Original train size: 4000
Test size: 1000
Number of features: 7
Using 50 fixed global evaluation samples

GLOBAL ATTRIBUTION | Train size = 500
RandomForestClassifier(n_estimators=200, random_state=42) | Train Acc: 1.0000 | Full Test Acc: 0.8660 | Eval Acc: 0.8800
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Saved: ./outputs_globalsoil_multisize/global_results_size_500.csv
  dataset_size  dataset_train_n  \
0          500              500   
1          500              500   
2          500              500   
3          500              500   
4          500              500   

                                               model      method  \
0  (DecisionTreeClassifier(max_features='sqrt', r...     shapley   
1  (DecisionTreeClassifier(max_features='sqrt', r...  least_core   
2  (DecisionTreeClassifier(max_features='sqrt', r...   nucleolus   
3  (DecisionTreeClassifier(max_features='sqrt', r...      gately   
4  (DecisionT

In [8]:
import os
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier


# Config

DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"

GLOBAL_RESULTS_DIR = "./outputs_globalsoil_multisize"

OUT_DIR = Path("./output_soil/quantus_evaluation_soil_global_multisize")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SEED = 123

DATA_SIZES = ["500", "1000", "full"]
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


# Wrapper

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1:
                out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()


# Data

def load_data():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")

    return X_train, y_train, X_test, y_test, feature_names


def get_train_subset(X_train_orig, y_train_orig, size_tag):
    if size_tag == "full":
        return X_train_orig, y_train_orig

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X_train_orig), size=int(size_tag), replace=False)

    return X_train_orig[idx], y_train_orig[idx]


def get_global_csv(size_tag):
    return os.path.join(
        GLOBAL_RESULTS_DIR,
        f"global_results_size_{size_tag}.csv"
    )


# Perturbation function

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [int(i) for i in idxs if int(i) < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        for b in range(arr.shape[0]):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        for b in range(arr.shape[0]):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert


# Manual aopc

def compute_aopc_manual(
    sk_model,
    X_test_2d,
    y_test,
    attributions_2d,
    top_k_list,
    baseline
):
    if not hasattr(sk_model, "predict_proba"):
        return np.nan

    base_probs = sk_model.predict_proba(X_test_2d)
    base_scores = base_probs[np.arange(len(y_test)), y_test]

    drops = []

    for k in top_k_list:
        if k > X_test_2d.shape[1]:
            continue

        X_perturbed = X_test_2d.copy()

        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]

        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]

        drops.append(np.mean(base_scores - new_scores))

    return float(np.mean(drops)) if len(drops) > 0 else np.nan


# Quantus metrics

def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(
            return_aggregate=True
        ),

        "complexity": quantus.Complexity(
            return_aggregate=True
        ),
    }

    return metrics


# Main

def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_data()

    all_rows = []

    for size_tag in DATA_SIZES:
        logger.info("=" * 80)
        logger.info(f"Evaluating GLOBAL results for train size: {size_tag}")
        logger.info("=" * 80)

        csv_path = get_global_csv(size_tag)

        if not os.path.exists(csv_path):
            logger.warning(f"Missing global result file: {csv_path}")
            continue

        df_global = pd.read_csv(csv_path)

        X_train, y_train = get_train_subset(
            X_train_orig,
            y_train_orig,
            size_tag
        )

        baseline = np.mean(X_train, axis=0)
        metrics = get_metrics(baseline)

        eval_indices = json.loads(df_global.iloc[0]["eval_indices"])

        X_eval = X_test[eval_indices]
        y_eval = y_test[eval_indices]

        x_batch = X_eval[:, np.newaxis, :]

        # Same model as Script 1

        model_name = "random_forest"

        logger.info(f"Model: {model_name}")

        sk_model = RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )

        sk_model.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, sk_model.predict(X_train))
        test_acc = accuracy_score(y_test, sk_model.predict(X_test))
        eval_acc = accuracy_score(y_eval, sk_model.predict(X_eval))

        logger.info(
            f"{model_name} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Full Test Acc: {test_acc:.4f} | "
            f"Eval Acc: {eval_acc:.4f}"
        )

        torch_model = SklearnTorchWrapper(sk_model).eval()

        df_model = df_global.copy()

        for method_name in df_model["method"].unique():
            logger.info(f"  Method: {method_name}")

            row = df_model[df_model["method"] == method_name].iloc[0]

            global_attr = np.array(
                json.loads(row["alloc_global"]),
                dtype=float
            )

            # Same global attribution vector is applied to every sample.
            a_batch_2d = np.tile(global_attr, (len(X_eval), 1))
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            result = {
                "setup": "global",
                "dataset_size": size_tag,
                "dataset_train_n": int(X_train.shape[0]),
                "n_eval_samples": int(len(X_eval)),
                "model": model_name,
                "method": method_name,
                "train_accuracy": float(train_acc),
                "test_accuracy": float(test_acc),
                "eval_accuracy": float(eval_acc),
            }

            for metric_name, metric_fn in metrics.items():
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_eval,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )

                    result[metric_name] = float(np.nanmean(scores))

                except Exception as e:
                    logger.warning(f"    {metric_name} error: {e}")
                    result[metric_name] = np.nan

            try:
                result["aopc_manual"] = compute_aopc_manual(
                    sk_model=sk_model,
                    X_test_2d=X_eval,
                    y_test=y_eval,
                    attributions_2d=a_batch_2d,
                    top_k_list=AOPC_TOPK_LIST,
                    baseline=baseline
                )

            except Exception as e:
                logger.warning(f"    AOPC error: {e}")
                result["aopc_manual"] = np.nan

            all_rows.append(result)

            logger.info(
                f"    FaithCorr={result.get('faithfulness_corr', np.nan):.4f}, "
                f"AOPC={result.get('aopc_manual', np.nan):.4f}"
            )

    df_out = pd.DataFrame(all_rows)

    out_csv = OUT_DIR / "quantussoil_results_global_multisize.csv"
    df_out.to_csv(out_csv, index=False)

    print("\nSaved global Quantus results to:")
    print(out_csv)

    print("\nFinal results:")
    print(df_out.to_string(index=False))


if __name__ == "__main__":
    main()

Original train size: 4000
Test size: 1000
Number of features: 7



Saved global Quantus results to:
output_soil/quantus_evaluation_soil_global_multisize/quantussoil_results_global_multisize.csv

Final results:
 setup dataset_size  dataset_train_n  n_eval_samples         model     method  train_accuracy  test_accuracy  eval_accuracy  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual
global          500              500              50 random_forest    shapley             1.0          0.866           0.88           0.405896               0.399618     0.434971           0.0        0.434971    0.531374    1.449637     0.305529
global          500              500              50 random_forest least_core             1.0          0.866           0.88           0.431420               0.403488     0.434971           0.0        0.434971    0.266632    1.821054     0.182500
global          500              500              50 random_forest  nucleolus             1.0          0.866          

In [11]:
import os
import json
import numpy as np
import pandas as pd

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error


# Config

DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"

GLOBAL_RESULTS_DIR = "./outputs_globalsoil_multisize"

DATA_SIZES = ["500", "1000", "full"]

OUT_CSV = "./outputs_globalsoil_multisize/global_vs_groundtruth_metrics.csv"


# Ground-truth SPI feature importances
GROUND_TRUTH = {
    "Organic Carbon": 0.064,
    "Nitrogen": 0.401,
    "Phosphorus": 0.047,
    "Potassium": 0.138,
    "pH": 0.138,
    "Zinc": 0.060,
    "Electrical Conductivity": 0.153,
}


# Load feature names

def load_feature_names(npz_path):
    data = np.load(npz_path, allow_pickle=True)

    if "feature_names" not in data:
        raise ValueError("feature_names not found in NPZ file.")

    return list(data["feature_names"])


# Metrics

def compute_metrics(attr_values, attr_feature_names, ground_truth_dict):
    attr_series = pd.Series(
        np.asarray(attr_values, dtype=float),
        index=list(attr_feature_names)
    )

    gt_series = pd.Series(ground_truth_dict, dtype=float)

    missing = [f for f in gt_series.index if f not in attr_series.index]
    extra = [f for f in attr_series.index if f not in gt_series.index]

    if missing:
        raise ValueError(f"Missing features in attribution: {missing}")

    if extra:
        print(f"Warning: extra attribution features ignored: {extra}")

    # Safe alignment by feature name
    attr_aligned = attr_series.loc[gt_series.index].values
    gt_aligned = gt_series.values

    pearson = pearsonr(attr_aligned, gt_aligned)[0]
    spearman = spearmanr(attr_aligned, gt_aligned)[0]
    cosine = cosine_similarity(
        attr_aligned.reshape(1, -1),
        gt_aligned.reshape(1, -1)
    )[0, 0]
    rmse = np.sqrt(mean_squared_error(gt_aligned, attr_aligned))

    return pearson, spearman, cosine, rmse


# Main

def main():
    feature_names = load_feature_names(DATA_NPZ_PATH)

    print("\nFeature order from NPZ:")
    for i, f in enumerate(feature_names):
        print(f"{i}: {f}")

    rows = []

    for size_tag in DATA_SIZES:
        csv_path = os.path.join(
            GLOBAL_RESULTS_DIR,
            f"global_results_size_{size_tag}.csv"
        )

        if not os.path.exists(csv_path):
            print(f"\nMissing file: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        for _, row in df.iterrows():
            alloc = np.array(json.loads(row["alloc_global"]), dtype=float)

            if len(alloc) != len(feature_names):
                raise ValueError(
                    f"Attribution length mismatch for size={size_tag}, "
                    f"method={row['method']}: "
                    f"len(alloc)={len(alloc)}, len(feature_names)={len(feature_names)}"
                )

            pearson, spearman, cosine, rmse = compute_metrics(
                attr_values=alloc,
                attr_feature_names=feature_names,
                ground_truth_dict=GROUND_TRUTH
            )

            rows.append({
                "setup": row.get("setup", "global"),
                "dataset_size": size_tag,
                "dataset_train_n": int(row["dataset_train_n"]),
                "model": "random_forest",
                "method": row["method"],
                "pearson": pearson,
                "spearman": spearman,
                "cosine": cosine,
                "rmse": rmse,
            })

    df_metrics = pd.DataFrame(rows)

    df_metrics.to_csv(OUT_CSV, index=False)

    print("\n===== Global Attribution vs Ground Truth =====")
    print(df_metrics.to_string(index=False))

    print(f"\nSaved to: {OUT_CSV}")


if __name__ == "__main__":
    main()


Feature order from NPZ:
0: Organic Carbon
1: Nitrogen
2: Phosphorus
3: Potassium
4: pH
5: Zinc
6: Electrical Conductivity

===== Global Attribution vs Ground Truth =====
 setup dataset_size  dataset_train_n         model     method  pearson  spearman   cosine     rmse
global          500              500 random_forest    shapley 0.911725  0.882919 0.950035 0.115372
global          500              500 random_forest least_core 0.897196  0.882919 0.331363 0.216686
global          500              500 random_forest  nucleolus 0.802734  0.882919 0.185195 0.208732
global          500              500 random_forest     gately 0.834704  0.882919 0.222607 0.208199
global          500              500 random_forest    banzhaf 0.904450  0.846881 0.940257 0.075916
global         1000             1000 random_forest    shapley 0.935755  0.882919 0.956862 0.135601
global         1000             1000 random_forest least_core 0.907588  0.774806 0.392796 0.212852
global         1000             1000 